# Azure Document Intelligence

## 1. What is Azure Document Intelligence?

**Azure AI Document Intelligence** is a managed Azure service that uses AI to **extract text, structure, tables, key-value pairs, and other information from documents** such as PDFs, invoices, forms, receipts, IDs, and business documents.

It is particularly useful when documents are **unstructured or semi-structured**.

```text
PDF / Image / Form
        ↓
Azure Document Intelligence
        ↓
Text + Tables + Fields + Structure
        ↓
JSON
        ↓
Application / RAG / Database
```

---

# 2. Why Do We Need It?

Consider a PDF invoice:

```text
Invoice Number: INV-1001

Customer: ABC Pvt Ltd

Total: ₹50,000

        Item        Qty     Price
        Laptop       2      80,000
        Monitor      1      20,000
```

A normal text extraction library may give you raw text.

Document Intelligence can identify the **document structure**:

```json
{
  "invoice_number": "INV-1001",
  "customer": "ABC Pvt Ltd",
  "total": 50000,
  "items": [
    {
      "item": "Laptop",
      "quantity": 2,
      "price": 80000
    }
  ]
}
```

That structured information can then be passed to your application, database, workflow, or LLM.

---

# 3. Main Capabilities

| Capability | Usage |
|---|---|
| **OCR / Read** | Extract text from scanned documents/images |
| **Layout** | Extract document structure, paragraphs, tables, sections |
| **Prebuilt Models** | Process common document types |
| **Custom Models** | Extract fields from organization-specific documents |
| **Tables** | Extract rows and columns |
| **Key-Value Pairs** | Extract fields such as `Name: Suraj` |
| **Selection Marks** | Extract checkboxes/selections |
| **Handwriting** | Extract handwritten text where supported |
| **Classification** | Identify document types |
| **Document Fields** | Return structured information |

---

# 4. OCR

OCR means:

> **Optical Character Recognition**

It converts text contained in images/scanned documents into machine-readable text.

Example:

```text
Scanned PDF
    ↓
OCR
    ↓
"Employee Name: Suraj"
```

This is especially important because a PDF can contain either:

- Actual text
- Scanned images of text

---

# 5. Read Model

The **Read** capability extracts text from documents and images.

```text
Image
 ↓
Read
 ↓
Text
```

Example:

```text
Input:
[Scanned employee certificate]

Output:
Employee Name: Suraj Khodade
Certificate: AWS Certified
Date: 15/05/2026
```

---

# 6. Layout Model ⭐⭐⭐⭐⭐

The Layout capability goes beyond simply extracting text.

It can identify elements such as:

```text
Document
 ├── Paragraphs
 ├── Headings
 ├── Tables
 ├── Selection marks
 └── Document structure
```

For RAG, this is valuable because you don't want to blindly flatten every PDF into plain text.

---

# 7. Prebuilt Models

Azure provides prebuilt models for common document scenarios.

Examples include:

| Prebuilt Model | Usage |
|---|---|
| Invoice | Extract invoice information |
| Receipt | Extract receipt information |
| ID documents | Extract identity information |
| Business cards | Extract contact information |
| Tax documents | Extract supported tax-form fields |
| Contracts / general documents | Depending on supported model capabilities |

Example:

```text
Invoice PDF
    ↓
Prebuilt Invoice Model
    ↓
Vendor
Invoice Number
Date
Total
Line Items
```

---

# 8. Custom Models ⭐⭐⭐⭐⭐

Suppose your organization has a custom form:

```text
Employee Joining Form

Employee ID: ______
Department: ______
Joining Date: ______
Manager: ______
```

A prebuilt invoice model isn't appropriate.

You can train/use a **custom document model** to extract your organization's specific fields.

```text
Custom Documents
      ↓
Training
      ↓
Custom Model
      ↓
New Document
      ↓
Structured Fields
```

---

# 9. Document Classification

Before extraction, you may need to determine what type of document you received.

For example:

```text
Incoming Document
       ↓
Classifier
       │
       ├── Invoice
       ├── Resume
       ├── Contract
       └── ID Document
```

Then route it to the appropriate processing model.

---

# 10. Tables

This is one of the important advantages over simple OCR.

Input:

```text
Product       Qty       Price
Laptop         2        80000
Monitor        3        30000
```

Document Intelligence can identify the table structure rather than returning only a flat sequence of words.

Conceptually:

```json
{
  "table": [
    ["Product", "Qty", "Price"],
    ["Laptop", "2", "80000"],
    ["Monitor", "3", "30000"]
  ]
}
```

---

# 11. Key-Value Pairs

Forms often contain:

```text
Name: Suraj
Department: Engineering
Location: Pune
```

Document Intelligence can identify relationships such as:

```text
Name       → Suraj
Department → Engineering
Location   → Pune
```

This is useful for structured extraction.

---

# 12. Azure Document Intelligence in RAG

This is particularly relevant to your **GenAI background**.

Suppose you have a 200-page PDF containing:

- Text
- Tables
- Scanned pages
- Forms

A simple pipeline could be:

```text
PDF
 ↓
Azure Document Intelligence
 ↓
Text + Tables + Structure
 ↓
Clean / Normalize
 ↓
Chunking
 ↓
Embeddings
 ↓
Azure AI Search
 ↓
Retriever
 ↓
Azure OpenAI
 ↓
Answer
```

So Document Intelligence is primarily the **document understanding/extraction layer**.

Azure AI Search is the **retrieval layer**.

Azure OpenAI is the **generation layer**.

---

# 13. Why Not Just Use PyPDF / Textract?

This is a good interview discussion.

### Simple PDF extraction

```text
PDF
 ↓
PyPDF
 ↓
Text
```

This works well when the PDF contains clean digital text.

But consider:

```text
Scanned PDF
+
Tables
+
Forms
+
Images
+
Complex Layout
```

A dedicated document AI service can provide richer document structure.

### AWS Comparison

Since you have used AWS:

> **Azure Document Intelligence is conceptually comparable to Amazon Textract for document/OCR extraction**, although their capabilities, APIs, models, and supported features differ.

| AWS | Azure |
|---|---|
| Amazon Textract | Azure Document Intelligence |
| OCR | OCR |
| Forms | Forms |
| Tables | Tables |
| Key-value pairs | Key-value pairs |
| Analyze documents | Analyze documents |

---

# 14. Simple Python Example

Using the Azure SDK:

```python
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

endpoint = "YOUR_ENDPOINT"
key = "YOUR_KEY"

client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

with open("invoice.pdf", "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-invoice",
        body=f
    )

result = poller.result()

for document in result.documents:
    for name, field in document.fields.items():
        print(name, field.content)
```

The important concept is:

```text
Document
    ↓
Document Intelligence Client
    ↓
Model
    ↓
Analysis
    ↓
Structured Result
```

---

# 15. Example: Layout Analysis

Conceptually:

```python
with open("document.pdf", "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-layout",
        body=f
    )

result = poller.result()

for page in result.pages:
    print(f"Page: {page.page_number}")

    for line in page.lines:
        print(line.content)
```

You can then transform the extracted content into chunks for your RAG pipeline.

---

# 16. Document Intelligence vs Azure AI Search

Very important distinction:

| Document Intelligence | Azure AI Search |
|---|---|
| Understands documents | Searches documents |
| OCR | Keyword search |
| Extracts tables | Vector search |
| Extracts fields | Hybrid search |
| Extracts structure | Semantic ranking |
| Document processing | Retrieval |
| Input → structured content | Query → relevant content |

Typical RAG architecture:

```text
PDF
 ↓
Document Intelligence
 ↓
Structured Content
 ↓
Azure AI Search
 ↓
Relevant Chunks
 ↓
Azure OpenAI
```

---

# 17. Document Intelligence vs Azure OpenAI

| Document Intelligence | Azure OpenAI |
|---|---|
| Document understanding | Generative AI |
| OCR | LLM |
| Tables | Text generation |
| Forms | Reasoning/generation |
| Field extraction | Summarization |
| Layout | Question answering |
| Structured extraction | Agentic workflows |

You can combine them:

```text
Document
 ↓
Document Intelligence
 ↓
Structured Content
 ↓
Azure OpenAI
 ↓
Summary / Extraction / Q&A
```

---

# 18. Production Considerations

For a production document-processing pipeline, consider:

### Large documents

Don't send a 200-page PDF directly to the LLM.

Use:

```text
Document Intelligence
        ↓
Chunking
        ↓
Embedding
        ↓
Search
        ↓
Relevant Context
        ↓
LLM
```

### Asynchronous processing

For large documents:

```text
Upload
 ↓
Queue
 ↓
Document Intelligence
 ↓
Processing
 ↓
Store result
 ↓
Index
```

### Validation

Extracted fields should be validated before being used downstream.

For example:

```python
class Invoice(BaseModel):
    invoice_number: str
    total: float
```

Then:

```text
Document Intelligence
        ↓
Pydantic Validation
        ↓
Business Validation
        ↓
Database / API
```

---

# 19. Common Interview Questions

### Q1. What is Azure Document Intelligence?

> "Azure Document Intelligence is a managed Azure service for extracting text, tables, fields, key-value pairs, and document structure from PDFs, images, forms, and other documents."

### Q2. What is OCR?

> "OCR converts text contained in scanned images or documents into machine-readable text."

### Q3. Prebuilt vs Custom Models?

> "Prebuilt models are designed for common document types such as invoices and receipts, while custom models are used when we need to extract organization-specific fields from custom document formats."

### Q4. How would you use Document Intelligence in RAG?

> "I would use Document Intelligence during ingestion to extract text, tables, and document structure. I would then clean and chunk the extracted content, generate embeddings, index it in Azure AI Search, and retrieve relevant content when answering questions with Azure OpenAI."

### Q5. Document Intelligence vs Azure AI Search?

> "Document Intelligence is primarily the document understanding and extraction layer, whereas Azure AI Search is the search and retrieval layer."

---

# 20. Senior-Level Scenario

### Interviewer:

> "You receive a 200-page PDF containing text, tables, scanned pages and images. How would you build a RAG pipeline?"

A strong architecture:

```text
                 200-page PDF
                       │
                       ▼
             Azure Document Intelligence
                       │
           ┌───────────┼───────────┐
           ▼           ▼           ▼
         Text        Tables      OCR
           │           │           │
           └───────────┼───────────┘
                       ▼
                 Normalize
                       │
                       ▼
             Structure-aware Chunking
                       │
                       ▼
                  Embeddings
                       │
                       ▼
               Azure AI Search
                       │
             Hybrid / Vector Search
                       │
                       ▼
                  Top-K Chunks
                       │
                       ▼
                 Azure OpenAI
                       │
                       ▼
                    Answer
```

### Interview explanation

> "I would first use Document Intelligence to extract text, tables, OCR content, and document structure. I would preserve page and section metadata because it's important for traceability. Then I would apply structure-aware chunking, generate embeddings, and index the chunks in Azure AI Search. At query time I would use hybrid or vector retrieval, optionally semantic ranking, and pass only the most relevant context to Azure OpenAI. This avoids putting the entire 200-page document into the LLM context."

That answer connects **Document Intelligence + RAG + Azure AI Search + Azure OpenAI** into one production architecture.